# M4.4 Magnetization and Bound Currents

Book pointer: chapter 13, sections 13.1 (magnetization and bound currents), 13.2 (Ampere's law for magnetic materials), 13.4 (boundary conditions).

## 1. Recall

1. Define the magnetization $\mathbf{M}$. Units. What microscopic quantity is it the density of?
2. Write the bound volume current $\mathbf{J}_b$ and the bound surface current $\mathbf{K}_b$ in terms of $\mathbf{M}$. Why does a uniform $\mathbf{M}$ give only a surface current?
3. Define $\mathbf{H}$ and state Ampere's law for $\mathbf{H}$ in differential and integral form. Which currents source $\mathbf{H}$, which source $\mathbf{B}$?
4. State the boundary conditions on $\mathbf{B}$ and $\mathbf{H}$ at a surface carrying free surface current $\mathbf{K}_f$. Which component of which field jumps, and by how much, at the surface of a magnet with no free current?
5. A long cylinder uniformly magnetized along its axis: what current distribution is it equivalent to, and what is $\mathbf{B}$ deep inside?

## 2. Principle to algorithm

Magnetization is a vector field on a grid like any other, so $\mathbf{J}_b = \nabla\times\mathbf{M}$ is the discrete curl from module 1, wrapped as `magnetization.bound_volume_current`. A surface current is what the curl produces where $\mathbf{M}$ steps to zero at the boundary of the body: on a grid the step is spread over one or two cells, the curl there is a spike, and integrating the spike through the surface recovers $\mathbf{K}_b = \mathbf{M}\times\hat{\mathbf{n}}$. This is the magnetic twin of M3.1, where $\rho_b = -\nabla\cdot\mathbf{P}$ and the spike integrated to $\sigma_b$.

Once the bound currents are known they are ordinary currents. The field of a magnetized body is the Biot-Savart field of its bound currents and nothing else. For a cylinder magnetized along its axis the bound current is a sheet $K_b = M$ running around the curved surface, which is a solenoid with $nI = M$, so the field comes from a stack of current loops built with `biot_savart.loop` and summed with `biot_savart.wire_B`. The grid route and the loop route must agree, and both must agree with the solenoid closed form on the axis. The auxiliary field $\mathbf{H} = \mathbf{B}/\mu_0 - \mathbf{M}$ then follows by subtraction, and the two boundary conditions ($B_n$ continuous, $B_t$ jumps by $\mu_0 K_b$) are measured on the computed field.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import grids, fields, biot_savart as bs, magnetization as mg
from emlib.biot_savart import MU0

plt.rcParams["figure.figsize"] = (6, 5)

### A uniformly magnetized cylinder on a grid

$\mathbf{M} = M_0\hat{\mathbf{z}}$ inside a cylinder of radius $R = 1$ and length $L = 2$, zero outside. Inside, $\nabla\times\mathbf{M} = 0$. At the curved surface $M_z$ drops to zero, so $(\nabla\times\mathbf{M})_\phi = -\partial M_z/\partial r$ is a spike running around the cylinder: that is the bound surface current $\mathbf{K}_b = \mathbf{M}\times\hat{\mathbf{r}} = M_0\hat{\boldsymbol\phi}$. At the flat end faces $M_z$ also drops to zero, but $\partial M_z/\partial z$ appears in no component of the curl, so there is no current there: $\mathbf{M}\times\hat{\mathbf{z}} = 0$. Along the $x$ axis the $\phi$ direction is $\pm\hat{\mathbf{y}}$, so the cell below reads off $J_y$ and integrates it through the surface.

In [ ]:
x, y, z, G = grids.grid3d((-2, 2), (-2, 2), (-2, 2), 81)
h = grids.spacing(x, y, z)
X, Y, Z = grids.components(G)
Rc = np.hypot(X, Y)
R, L, M0 = 1.0, 2.0, 1e3                                   # M0 in A/m
inside = (Rc < R) & (np.abs(Z) < L / 2)
Mag = np.zeros_like(G); Mag[..., 2] = M0 * inside

Jb = mg.bound_volume_current(Mag, *h)
k = 40                                                      # the plane z = 0
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
im = axes[0].imshow(grids.magnitude(Jb)[:, :, k].T, origin="lower", extent=(-2, 2, -2, 2), cmap="magma"); plt.colorbar(im, ax=axes[0], label="|J_b|  [A/m^2]")
axes[0].set_title("|curl M| in the plane z = 0: a ring at r = R"); axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
axes[1].plot(x, Jb[:, 40, k, 1], "o-", ms=3); axes[1].set_xlabel("x"); axes[1].set_ylabel("J_y along the x axis  [A/m^2]"); axes[1].set_title("the surface current is a spike")
plt.tight_layout(); plt.show()

right = (x > 0.5) & (x < 1.5); left = (x < -0.5) & (x > -1.5)
print(f"integral of J_y through the surface at x = +R: {np.trapezoid(Jb[right, 40, k, 1], dx=h[0]) / M0:+.4f} M0   (K_b = M x r_hat = +M0 phi_hat, and phi_hat = +y_hat there)")
print(f"integral of J_y through the surface at x = -R: {np.trapezoid(Jb[left, 40, k, 1], dx=h[0]) / M0:+.4f} M0   (phi_hat = -y_hat there)")
print(f"from the formula: {mg.bound_surface_current([0, 0, M0], [1, 0, 0]) / M0} M0 at +R,  {mg.bound_surface_current([0, 0, M0], [-1, 0, 0]) / M0} M0 at -R")
print(f"max |J_b| on the axis through the end faces (x = y = 0): {np.abs(Jb[40, 40, :, :]).max():.1e} A/m^2   (no current on the flat faces)")
print(f"max |J_b| well inside (r < 0.8R, |z| < 0.8L/2): {np.abs(Jb[(Rc < 0.8 * R) & (np.abs(Z) < 0.8)]).max():.1e} A/m^2")

### A non-uniform magnetization

$\mathbf{M} = M_0\,(1 - r^2/R^2)\,\hat{\mathbf{z}}$ inside the same cylinder. Now the curl does not vanish inside: $(\nabla\times\mathbf{M})_\phi = -\partial M_z/\partial r = 2M_0 r/R^2$, a volume current circulating around the axis and growing linearly with $r$. Because $M_z$ goes continuously to zero at $r = R$, the surface current $\mathbf{M}\times\hat{\mathbf{n}}$ is zero there, and the grid shows no spike.

In [ ]:
Mag2 = np.zeros_like(G); Mag2[..., 2] = M0 * (1 - Rc**2 / R**2) * inside
Jb2 = mg.bound_volume_current(Mag2, *h)
Jphi_exact = 2 * M0 * Rc / R**2 * inside
# J_phi = -J_x sin(phi) + J_y cos(phi)
phi = np.arctan2(Y, X)
Jphi = -Jb2[..., 0] * np.sin(phi) + Jb2[..., 1] * np.cos(phi)

plt.plot(x, Jphi[:, 40, k] / M0, "o", ms=3, label="grid: J_phi along the x axis")
plt.plot(x, Jphi_exact[:, 40, k] / M0, "k-", lw=1, label="2 M0 r / R^2 inside, 0 outside")
plt.xlabel("x"); plt.ylabel("J_phi / M0  [1/m]"); plt.legend(); plt.title("volume bound current of M = M0 (1 - r^2/R^2) z_hat"); plt.show()
core = (Rc < 0.8 * R) & (np.abs(Z) < 0.8)
print(f"max |J_phi - exact| / max J_phi, inside r < 0.8R: {np.abs(Jphi - Jphi_exact)[core].max() / Jphi_exact.max():.2e}")
i_R = np.argmin(np.abs(x - R))
print(f"J_phi at r = R: {Jphi[i_R, 40, k] / (M0 / R):.3f} M0/R, between the interior slope value 2 M0/R and the exterior 0: a kink in M, not a step, so no sheet current")

### The field of the magnetized cylinder: bound currents are ordinary currents

The uniform cylinder carries the sheet current $K_b = M_0$ on its curved surface, which is a solenoid with $nI = M_0$. Two ways to compute its field. The loop route: slice the surface into $N$ loops of radius $R$ at heights $z_k$, each carrying $I = M_0\,\Delta z$, and sum their Biot-Savart fields. The grid route: take the $\mathbf{J}_b$ computed above and treat every cell as a current element $\mathbf{J}_b\,h^3 = I\,d\mathbf{l}$, which `wire_B` accepts as a segment with $d\mathbf{l} = \mathbf{J}_b h^3 / I$. On the axis both must agree with `solenoid_axis_B` with $n I = M_0$. The grid route comes out about one percent low at the centre and a few percent off near the end faces, because the ring of cells is a staircase approximation to the circle: the current elements sit at cell centres one grid step either side of $r = R$ and flow along grid directions rather than along the circle.

In [ ]:
def cylinder_loops(R, L, M, n_loops, n_seg=120):
    '''Segments of a stack of n_loops current loops representing K_b = M on a cylinder of radius R, length L. Returns (mid, dl, I).'''
    dz = L / n_loops
    zk = -L / 2 + (np.arange(n_loops) + 0.5) * dz
    mids, dls = zip(*[bs.segments(bs.loop(R, n_seg, centre=(0, 0, zc))) for zc in zk])
    return np.vstack(mids), np.vstack(dls), M * dz

mid, dl, I = cylinder_loops(R, L, M0, 40)
zz = np.linspace(-3, 3, 121)
axis = np.stack([0 * zz, 0 * zz, zz], axis=-1)
B_loops = bs.wire_B(mid, dl, I, axis)[:, 2]
B_exact = bs.solenoid_axis_B(R, L, 1.0, M0, zz)             # n I = M0

# Grid route: every cell with current is a segment I dl = J h^3.
cell = h[0] * h[1] * h[2]
has_J = grids.magnitude(Jb) > 0
mid_g = G[has_J]; dl_g = Jb[has_J] * cell                   # (M, 3) with I = 1 A
B_grid = bs.wire_B(mid_g, dl_g, 1.0, axis)[:, 2]
print(f"{has_J.sum()} grid cells carry bound current")

plt.plot(zz, B_exact / (MU0 * M0), "k-", lw=3, alpha=0.3, label="solenoid closed form, n I = M0")
plt.plot(zz, B_loops / (MU0 * M0), "C1.", ms=4, label=f"{40} loops of I = M0 dz")
plt.plot(zz, B_grid / (MU0 * M0), "C0x", ms=4, label="grid J_b as current elements")
plt.axvline(-L / 2, color="grey", ls=":"); plt.axvline(L / 2, color="grey", ls=":")
plt.xlabel("z on the axis"); plt.ylabel("B_z / (mu0 M0)"); plt.legend(); plt.title("field of a uniformly magnetized cylinder, R = 1, L = 2"); plt.show()
print(f"B_z at the centre: loops {B_loops[60] / (MU0 * M0):.4f} mu0 M0, grid {B_grid[60] / (MU0 * M0):.4f} mu0 M0, closed form {B_exact[60] / (MU0 * M0):.4f} mu0 M0  (= L / sqrt(L^2 + 4R^2) for a finite solenoid)")
print(f"max relative error on the axis: loops {np.abs(B_loops / B_exact - 1).max():.1e}, grid {np.abs(B_grid / B_exact - 1).max():.1e}")

In [ ]:
# The field map in the plane y = 0: a bar magnet.
xs = np.linspace(-3, 3, 121); zs = np.linspace(-3, 3, 121)
Xp, Zp = np.meshgrid(xs, zs, indexing="ij")
plane = np.stack([Xp, np.zeros_like(Xp), Zp], axis=-1)
Bp = bs.wire_B(mid, dl, I, plane)
Bmag = grids.magnitude(Bp)
fig, ax = plt.subplots(figsize=(6, 6))
ax.streamplot(xs, zs, Bp[..., 0].T, Bp[..., 2].T, color=np.log10(Bmag / (MU0 * M0)).T, cmap="viridis", density=1.6, linewidth=0.8)
ax.add_patch(plt.Rectangle((-R, -L / 2), 2 * R, L, fill=False, lw=2, color="C3"))
ax.set_aspect("equal"); ax.set_xlabel("x"); ax.set_ylabel("z"); ax.set_title("B of the magnetized cylinder (colour: log10 B / mu0 M0)")
plt.show()

### H along the axis

$\mathbf{H} = \mathbf{B}/\mu_0 - \mathbf{M}$. Outside the magnet $\mathbf{M} = 0$ and $\mathbf{H} = \mathbf{B}/\mu_0$. Inside, $B_z < \mu_0 M_0$ for any finite cylinder (the closed form gives $L/\sqrt{L^2 + 4R^2}$ at the centre), so $H_z = B_z/\mu_0 - M_0$ is negative: $\mathbf{H}$ points against $\mathbf{M}$ inside the magnet. This is the demagnetizing field. There is no free current anywhere, so $\nabla\times\mathbf{H} = 0$ everywhere, and $\nabla\cdot\mathbf{H} = -\nabla\cdot\mathbf{M}$ puts the sources of $\mathbf{H}$ on the end faces where $M_z$ steps: the magnetic-pole picture. On the axis $B_z$ is continuous through the end face while $M_z$ jumps by $M_0$, so $H_z$ jumps by $-M_0$ there.

In [ ]:
Mz_axis = M0 * (np.abs(zz) < L / 2)
H_axis = mg.H_field(np.stack([0 * zz, 0 * zz, B_loops], axis=-1), np.stack([0 * zz, 0 * zz, Mz_axis], axis=-1))[:, 2]
plt.plot(zz, B_loops / MU0 / M0, label="B_z / mu0")
plt.plot(zz, Mz_axis / M0, "k--", label="M_z")
plt.plot(zz, H_axis / M0, "C3", label="H_z = B_z / mu0 - M_z")
plt.axhline(0, color="grey", lw=0.5); plt.axvline(-L / 2, color="grey", ls=":"); plt.axvline(L / 2, color="grey", ls=":")
plt.xlabel("z on the axis"); plt.ylabel("in units of M0"); plt.legend(); plt.title("H is demagnetizing inside, equals B / mu0 outside"); plt.show()
print(f"at the centre: B_z / mu0 = {B_loops[60] / MU0 / M0:+.4f} M0, H_z = {H_axis[60] / M0:+.4f} M0")
j_out = np.argmin(np.abs(zz - 1.1)); j_in = np.argmin(np.abs(zz - 0.9))
print(f"just inside the end face (z = 0.9): B_z / mu0 = {B_loops[j_in] / MU0 / M0:+.4f} M0, H_z = {H_axis[j_in] / M0:+.4f} M0")
print(f"just outside (z = 1.1):            B_z / mu0 = {B_loops[j_out] / MU0 / M0:+.4f} M0, H_z = {H_axis[j_out] / M0:+.4f} M0")

### Exercise: bound surface current of a magnetized sphere

A sphere of radius $R$ uniformly magnetized along $z$ has $\mathbf{K}_b = \mathbf{M}\times\hat{\mathbf{r}} = M\sin\theta\,\hat{\boldsymbol\phi}$: a current sheet that is strongest at the equator and vanishes at the poles. Replace the body of `sphere_surface_current_loops` so it represents that sheet as $n$ loops at polar angles $\theta_k$ (midpoints of $n$ equal bands), each of radius $R\sin\theta_k$ at height $R\cos\theta_k$, carrying $I_k = M\sin\theta_k\,R\,\Delta\theta$. `wire_B` takes one current per call, so the check sums the loops one at a time. The field inside a uniformly magnetized sphere is uniform, $\mathbf{B} = \tfrac23\mu_0\mathbf{M}$, which the check requires at the centre and at an off-centre interior point.

In [ ]:
def sphere_surface_current_loops(R, M, n, n_seg=120):
    '''Return (mids, dls, currents): one (mid, dl) pair per loop and the loop current, for K_b = M sin(theta) phi_hat.'''
    dtheta = np.pi / n                                                   # replace from here with your own version
    theta = (np.arange(n) + 0.5) * dtheta
    mids, dls, currents = [], [], []
    for th in theta:
        m_, d_ = bs.segments(bs.loop(R * np.sin(th), n_seg, centre=(0, 0, R * np.cos(th))))
        mids.append(m_); dls.append(d_); currents.append(M * np.sin(th) * R * dtheta)
    return mids, dls, np.array(currents)


def sphere_B(R, M, n, obs):
    mids, dls, currents = sphere_surface_current_loops(R, M, n)
    return sum(bs.wire_B(m_, d_, I_, obs) for m_, d_, I_ in zip(mids, dls, currents))


pts = np.array([[0.0, 0.0, 0.0], [0.3, 0.2, -0.4]])
B_sph = sphere_B(1.0, M0, 100, pts)
print(f"B_z at the centre and at an interior point: {B_sph[:, 2] / (MU0 * M0)} mu0 M0   expected 2/3 = {2 / 3:.4f}")
print(f"transverse components: {np.abs(B_sph[:, :2]).max() / (MU0 * M0):.1e} mu0 M0")
assert np.allclose(B_sph[:, 2], 2 / 3 * MU0 * M0, rtol=1e-3) and np.abs(B_sph[:, :2]).max() < 1e-3 * MU0 * M0

## 4. Verify

Three closed-form checks on the cylinder's field, all from the loop stack.

A long cylinder ($L = 20R$) has $B_z = \mu_0 M_0$ at its centre to within $1 - L/\sqrt{L^2 + 4R^2} = 0.5\%$, which is the field inside an infinite solenoid with $nI = M_0$.

The boundary conditions, measured. Across the curved surface, where the sheet current $K_b = M_0$ flows, the tangential component $B_z$ must jump by $\mu_0 K_b = \mu_0 M_0$ while the normal component $B_r$ is continuous. Across the flat end face there is no surface current, so both components are continuous. The jump condition is a statement about the limit of vanishing distance from the surface. Sampling at $r = R \pm \delta$ measures the jump plus $2\delta$ times the field's slope, so the table lists the difference for several $\delta$: a true jump tends to a constant, while a steep but continuous field gives a difference that halves when $\delta$ halves. Extrapolating the two smallest $\delta$ to zero, $2D(\delta) - D(2\delta)$, removes the slope term. The loop stack is used rather than the grid currents because the staircase error of the grid ring is larger than the effect being measured; its loop spacing is kept well below $\delta$ so the stack looks like a smooth sheet from the sample points.

In [ ]:
# Long cylinder: B at the centre.
mid_l, dl_l, I_l = cylinder_loops(R, 20 * R, M0, 400)
B_long = bs.wire_B(mid_l, dl_l, I_l, np.array([[0.0, 0.0, 0.0]]))[0, 2]
print(f"long cylinder, L = 20R: B_z(centre) = {B_long / (MU0 * M0):.4f} mu0 M0   closed form {20 / np.sqrt(20**2 + 4):.4f}")
assert abs(B_long / (MU0 * M0) - 1) < 1e-2

# Boundary conditions on the L = 2R cylinder, with a fine stack: loop spacing 0.0125 R and segments of 0.006 R,
# both well below the smallest sampling distance so that the stack looks like a smooth sheet from the sample points.
mid_f, dl_f, I_f = cylinder_loops(R, L, M0, 160, n_seg=1000)
print("\ndifferences across the surface, in units of mu0 M0")
print("   delta   curved surface at z = 0.5R: B_z(R-d) - B_z(R+d)   B_r(R-d) - B_r(R+d)   |   end face at r = 0.5R: B_z(L/2-d) - B_z(L/2+d)")
D = {}
for d in (0.2, 0.1, 0.05, 0.025):
    pts = np.array([[R - d, 0, 0.5 * R], [R + d, 0, 0.5 * R], [R / 2, 0, L / 2 - d], [R / 2, 0, L / 2 + d]])
    Bq = bs.wire_B(mid_f, dl_f, I_f, pts) / (MU0 * M0)
    D[d] = (Bq[0, 2] - Bq[1, 2], Bq[0, 0] - Bq[1, 0], Bq[2, 2] - Bq[3, 2])
    print(f"  {d:6.3f}                      {D[d][0]:+8.4f}              {D[d][1]:+8.4f}                        {D[d][2]:+8.4f}")
jump = [2 * D[0.025][i] - D[0.05][i] for i in range(3)]
print(f"  extrapolated to delta = 0:  {jump[0]:+8.4f}              {jump[1]:+8.4f}                        {jump[2]:+8.4f}   expected +1, 0, 0")
assert abs(jump[0] - 1.0) < 0.02 and abs(jump[1]) < 0.01 and abs(jump[2]) < 0.01

# The profile of B across the curved surface at z = 0.5R.
rr = np.linspace(0, 2, 201)
Br_line = bs.wire_B(mid_f, dl_f, I_f, np.stack([rr, 0 * rr, 0.5 * R + 0 * rr], axis=-1))
plt.plot(rr, Br_line[:, 2] / (MU0 * M0), label="B_z (tangential to the curved surface)"); plt.plot(rr, Br_line[:, 0] / (MU0 * M0), label="B_r (normal)")
plt.axvline(R, color="grey", ls=":"); plt.xlabel("r at z = 0.5R"); plt.ylabel("B / (mu0 M0)"); plt.legend(); plt.title("B_z jumps by mu0 M0 at the current sheet, B_r is continuous"); plt.show()

## 5. Explore and rebuild

Explore. Shorten the cylinder to a disk magnet, $L = R/2$, and compute $B_z$ on the axis from $z = L$ out to $z = 20R$. Far away the magnet is a dipole with moment $m = M_0\,\pi R^2 L$ (magnetization times volume), so $B_z \to \mu_0 m/2\pi z^3$. Plot the ratio of the computed field to the dipole law against $z/R$ and find where it reaches 1 within a percent. Then do the same for the $L = 2R$ cylinder and explain, in terms of the distance between the two faces where $\mathbf{H}$ has its sources, why the longer magnet takes longer to look like a dipole.

Rebuild. Close this notebook and write `bound_volume_current`, the recovery of $K_b$ from the spike, and the loop-stack field of a magnetized cylinder from a blank editor, then reproduce the centre value $L/\sqrt{L^2 + 4R^2}$.